In [2]:
#(1) Install JDK

!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,913 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,613 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:

In [3]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

In [4]:
#(2) Install Spark

!wget -q https://dlcdn.apache.org/spark/spark-4.0.0/spark-3.5.6-bin-hadoop3.tgz

# unzip the spark file to the current folder
!tar xf spark-3.5.6-bin-hadoop3.tgz


#Check this site for the latest download link https://spark.apache.org/downloads.html

tar: spark-3.5.6-bin-hadoop3.tgz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now


In [5]:
#(3) Install pyspark

!pip install -q findspark
!pip install pyspark
!pip install py4j


In [44]:
#(4) Setup Environment variables

import os
import sys
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.6-bin-hadoop3"



In [45]:
#import findspark
#findspark.init()
#findspark.find()


In [11]:
!rm -rf spark-*

In [12]:
(2) # Install Spark — robust download
!rm -f spark-3.5.6-bin-hadoop3.tgz

!curl -L --retry 5 --retry-delay 5 \
  -o spark-3.5.6-bin-hadoop3.tgz \
  https://archive.apache.org/dist/spark/spark-3.5.6/spark-3.5.6-bin-hadoop3.tgz

!tar -xzf spark-3.5.6-bin-hadoop3.tgz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  382M  100  382M    0     0  10.0M      0  0:00:37  0:00:37 --:--:-- 6230k


In [13]:
# (3) Install pyspark
!pip install -q pyspark==3.5.6 py4j findspark

In [14]:
# (4) Setup Environment variables
import findspark
findspark.init("/content/spark-3.5.6-bin-hadoop3")

In [15]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("week 6b inclass activity DF").getOrCreate()
spark

In [16]:
import findspark
findspark.init()
findspark.find()

'/content/spark-3.5.6-bin-hadoop3'



---


# ***Note: Each Question is for 3 points***

---



## Task-0: CREATE Spark Session with appropriate App name.

In [17]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("CIS660 Assignment 6").getOrCreate()
spark

# Task-1: Download Google play store file, available on Blackboard (under Datasets)

### Q1: Load Google play store file into a dataframe (df1), and  

1.   print its schema, and
2.   sample (5) records, and
3.   count number of rows

In [18]:
df1 = spark.read.csv("/googleplaystore.csv", header=True, inferSchema=True)
df1.printSchema()
df1.show(5)
df1.count()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)

+--------------------+--------------+------+-------+----+-----------+----+-----+--------------+--------------------+----------------+------------------+------------+
|                 App|      Category|Rating|Reviews|Size|   Installs|Type|Price|Content Rating|              Genres|    Last Updated|       Current Ver| Android Ver|
+--------------------+--------------+------+-------+----+-----------+----+-----+--------------+--------------------+----------------+--------------

10841

### Q2: Change the data-types of Rating, Reviews and Price to Integer

In [19]:
from pyspark.sql.functions import col
df1 = df1.withColumn("Rating", col("Rating").cast("integer")) \
         .withColumn("Reviews", col("Reviews").cast("integer")) \
         .withColumn("Price", col("Price").cast("integer"))
df1.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: integer (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



### Q3. Convert the dataframe into SQL view/temporary table

In [20]:
df1.createOrReplaceTempView("googleplaystore")

### Q4: Count the number of apps in the 'Tools' genre which have more than 100 reviews.



```
(a) Solve using DF functions
```



In [21]:
df1.filter((df1["Genres"] == "Tools") & (df1["Reviews"] > 100)).count()

535

```
(b) Solve using SQL Query
```


In [22]:
spark.sql("SELECT COUNT(*) FROM googleplaystore WHERE Genres = 'Tools' AND Reviews > 100").show()

+--------+
|count(1)|
+--------+
|     535|
+--------+



### Q5: Select all apps that contain the word 'book' in the title



```
(a) Solve using DF functions
```



In [23]:
df1.filter(df1["App"].contains("book")).select("App").show(truncate=False)

+------------------------------------------------+
|App                                             |
+------------------------------------------------+
|Coloring book moana                             |
|Download free book with green book              |
|English Grammar Complete Handbook               |
|AlReader -any text book reader                  |
|ReadEra – free ebook reader                     |
|Ebook Reader                                    |
|Litnet - E-books                                |
|Read books online                               |
|eBoox: book reader fb2 epub zip                 |
|Flybook                                         |
|eBoox new: Reader for fb2 epub zip books        |
|Facebook Pages Manager                          |
|Facebook Ads Manager                            |
|Teacher's Gradebook - Additio                   |
|My Recipes Cookbook : RecetteTek                |
|Cookbook Recipes                                |
|TheFork - Restaurants booking 

```
(b) Solve using SQL Query
```


In [24]:
spark.sql("SELECT App FROM googleplaystore WHERE App LIKE '%book%'").show(truncate=False)

+------------------------------------------------+
|App                                             |
+------------------------------------------------+
|Coloring book moana                             |
|Download free book with green book              |
|English Grammar Complete Handbook               |
|AlReader -any text book reader                  |
|ReadEra – free ebook reader                     |
|Ebook Reader                                    |
|Litnet - E-books                                |
|Read books online                               |
|eBoox: book reader fb2 epub zip                 |
|Flybook                                         |
|eBoox new: Reader for fb2 epub zip books        |
|Facebook Pages Manager                          |
|Facebook Ads Manager                            |
|Teacher's Gradebook - Additio                   |
|My Recipes Cookbook : RecetteTek                |
|Cookbook Recipes                                |
|TheFork - Restaurants booking 

### Q6: For each Category, compute min, max, sum, avg, and total (count) number of rating.  Sort the result in descending order of the count



```
(a) Solve using DF functions
```



In [25]:
from pyspark.sql.functions import min, max, sum, avg, count, desc
df1.groupBy("Category") \
   .agg(min("Rating").alias("min_rating"),
        max("Rating").alias("max_rating"),
        sum("Rating").alias("sum_rating"),
        avg("Rating").alias("avg_rating"),
        count("Rating").alias("count_rating")) \
   .orderBy(desc("count_rating")) \
   .show()

+-------------------+----------+----------+----------+------------------+------------+
|           Category|min_rating|max_rating|sum_rating|        avg_rating|count_rating|
+-------------------+----------+----------+----------+------------------+------------+
|             FAMILY|         1|         5|      6593|  3.77389811104751|        1747|
|               GAME|         1|         5|      4241|3.8659981768459435|        1097|
|              TOOLS|         1|         5|      2671| 3.638964577656676|         734|
|       PRODUCTIVITY|         1|         5|      1338|3.8119658119658117|         351|
|            MEDICAL|         1|         5|      1320|3.7714285714285714|         350|
|      COMMUNICATION|         1|         5|      1250|3.8109756097560976|         328|
|            FINANCE|         1|         5|      1197|3.7058823529411766|         323|
|             SPORTS|         1|         5|      1220| 3.824451410658307|         319|
|        PHOTOGRAPHY|         2|         5|

```
(b) Solve using SQL Query
```


In [26]:
spark.sql("""
SELECT Category,
       MIN(Rating) AS min_rating,
       MAX(Rating) AS max_rating,
       SUM(Rating) AS sum_rating,
       AVG(Rating) AS avg_rating,
       COUNT(Rating) AS count_rating
FROM googleplaystore
GROUP BY Category
ORDER BY count_rating DESC
""").show()

+-------------------+----------+----------+----------+------------------+------------+
|           Category|min_rating|max_rating|sum_rating|        avg_rating|count_rating|
+-------------------+----------+----------+----------+------------------+------------+
|             FAMILY|         1|         5|      6593|  3.77389811104751|        1747|
|               GAME|         1|         5|      4241|3.8659981768459435|        1097|
|              TOOLS|         1|         5|      2671| 3.638964577656676|         734|
|       PRODUCTIVITY|         1|         5|      1338|3.8119658119658117|         351|
|            MEDICAL|         1|         5|      1320|3.7714285714285714|         350|
|      COMMUNICATION|         1|         5|      1250|3.8109756097560976|         328|
|            FINANCE|         1|         5|      1197|3.7058823529411766|         323|
|             SPORTS|         1|         5|      1220| 3.824451410658307|         319|
|        PHOTOGRAPHY|         2|         5|

### Q7: Drop the "Content Rating" column from the dataframe.

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [29]:
df1.groupBy("Type").count().show()

+------+-----+
|  Type|count|
+------+-----+
|     0|    1|
|102248|    1|
|   NaN|    1|
|  Free|10037|
|  Paid|  800|
|  2509|    1|
+------+-----+



### Q8: Add a new column called "**Number_of_Installs**". Copy the value from

*   Installs column into this new column but exclude the comma (,) and plus (+) sign.
*   For example:10,000+ should be stored as 10000



In [30]:
spark.sql("SELECT Type, COUNT(*) FROM googleplaystore GROUP BY Type").show()

+------+--------+
|  Type|count(1)|
+------+--------+
|     0|       1|
|102248|       1|
|   NaN|       1|
|  Free|   10037|
|  Paid|     800|
|  2509|       1|
+------+--------+



###Q9: Rename the last three columns and replace the empty space with underscore(_).
1. 'Last Updated' to 'Last_Updated'
2. 'Current Ver' to 'Current_Ver'
3. 'Android Ver' to 'Android_Ver'

### Q10: Use "When Otherwise" to create a new column called "Ranks". Based on the ratings, compute the ranks for each app as follow:

**Rating Range	Category**
1. 4.0 to 5.0 –	High Rank
2. 3.0 to 3.9 –	Moderate Rank
3. 2.0 to 2.9 –	Low Rank
4. 1.0 to 1.9 –	Very Low Rank
5. 0.0 to 0.9 –	Unacceptable Rank


In [31]:
from pyspark.sql.functions import when
df1 = df1.withColumn("Ranks",
                     when((col("Rating") >= 4.0) & (col("Rating") <= 5.0), "High Rank")
                     .when((col("Rating") >= 3.0) & (col("Rating") <= 3.9), "Moderate Rank")
                     .when((col("Rating") >= 2.0) & (col("Rating") <= 2.9), "Low Rank")
                     .when((col("Rating") >= 1.0) & (col("Rating") <= 1.9), "Very Low Rank")
                     .when((col("Rating") >= 0.0) & (col("Rating") <= 0.9), "Unacceptable Rank")
                     .otherwise("Unknown"))
df1.select("App", "Rating", "Ranks").show()

+--------------------+------+-------------+
|                 App|Rating|        Ranks|
+--------------------+------+-------------+
|Photo Editor & Ca...|     4|    High Rank|
| Coloring book moana|     3|Moderate Rank|
|U Launcher Lite –...|     4|    High Rank|
|Sketch - Draw & P...|     4|    High Rank|
|Pixel Draw - Numb...|     4|    High Rank|
|Paper flowers ins...|     4|    High Rank|
|Smoke Effect Phot...|     3|Moderate Rank|
|    Infinite Painter|     4|    High Rank|
|Garden Coloring Book|     4|    High Rank|
|Kids Paint Free -...|     4|    High Rank|
|Text on Photo - F...|     4|    High Rank|
|Name Art Photo Ed...|     4|    High Rank|
|Tattoo Name On My...|     4|    High Rank|
|Mandala Coloring ...|     4|    High Rank|
|3D Color Pixel by...|     4|    High Rank|
|Learn To Draw Kaw...|     3|Moderate Rank|
|Photo Designer - ...|     4|    High Rank|
|350 Diy Room Deco...|     4|    High Rank|
|FlipaClip - Carto...|     4|    High Rank|
|        ibis Paint X|     4|   

###Q11: Solve Q10 again, but using SQL's expr() function, and WHEN-THEN statement

In [32]:
from pyspark.sql.functions import expr
df1 = df1.withColumn("Ranks", expr("""
CASE
    WHEN Rating >= 4.0 AND Rating <= 5.0 THEN 'High Rank'
    WHEN Rating >= 3.0 AND Rating <= 3.9 THEN 'Moderate Rank'
    WHEN Rating >= 2.0 AND Rating <= 2.9 THEN 'Low Rank'
    WHEN Rating >= 1.0 AND Rating <= 1.9 THEN 'Very Low Rank'
    WHEN Rating >= 0.0 AND Rating <= 0.9 THEN 'Unacceptable Rank'
    ELSE 'Unknown'
END
"""))
df1.select("App", "Rating", "Ranks").show()

+--------------------+------+-------------+
|                 App|Rating|        Ranks|
+--------------------+------+-------------+
|Photo Editor & Ca...|     4|    High Rank|
| Coloring book moana|     3|Moderate Rank|
|U Launcher Lite –...|     4|    High Rank|
|Sketch - Draw & P...|     4|    High Rank|
|Pixel Draw - Numb...|     4|    High Rank|
|Paper flowers ins...|     4|    High Rank|
|Smoke Effect Phot...|     3|Moderate Rank|
|    Infinite Painter|     4|    High Rank|
|Garden Coloring Book|     4|    High Rank|
|Kids Paint Free -...|     4|    High Rank|
|Text on Photo - F...|     4|    High Rank|
|Name Art Photo Ed...|     4|    High Rank|
|Tattoo Name On My...|     4|    High Rank|
|Mandala Coloring ...|     4|    High Rank|
|3D Color Pixel by...|     4|    High Rank|
|Learn To Draw Kaw...|     3|Moderate Rank|
|Photo Designer - ...|     4|    High Rank|
|350 Diy Room Deco...|     4|    High Rank|
|FlipaClip - Carto...|     4|    High Rank|
|        ibis Paint X|     4|   

###Q12: Select and display only the following columns: App, Rating, Reviews,Size in ascending order of ratings.

In [33]:
df1.select("App", "Rating", "Reviews", "Size").orderBy("Rating").show()

+--------------------+------+-------+----+
|                 App|Rating|Reviews|Size|
+--------------------+------+-------+----+
|Random Video Chat...|  NULL|      3|4.8M|
|i miss you quotes...|  NULL|      0|5.0M|
|Meet With Strange...|  NULL|      2|3.7M|
|Wrinkles and reju...|  NULL|    182|5.7M|
|Ost. Zombies Cast...|  NULL|      1|4.6M|
|Skin Care and Nat...|  NULL|    654|7.4M|
|  Dating White Girls|  NULL|      0|3.6M|
|Recipes and tips ...|  NULL|     35|3.1M|
|        Geeks Dating|  NULL|      0| 13M|
|Anonymous caller ...|  NULL|    161|2.7M|
|Live chat - free ...|  NULL|      1|8.7M|
|URBANO V 02 instr...|  NULL|    114|7.3M|
|  CAM5678 Video Chat|  NULL|      0| 39M|
|      Toronto Dating|  NULL|      0| 14M|
|Video chat live a...|  NULL|      0|8.0M|
|Private Dating, H...|  NULL|      0| 18k|
|      chat live chat|  NULL|     24|3.9M|
|   Random Video Chat|  NULL|      3| 16M|
|   Pet Lovers Dating|  NULL|      0| 14M|
|Manicure - nail d...|  NULL|    119|3.7M|
+----------

###Q13: Filter the dataframe to remove all the rows where review count is less than 10K (reviews < 10000)

In [34]:
df1 = df1.filter(df1["Reviews"] >= 10000)
df1.show()

+--------------------+-----------------+------+-------+------------------+-----------+----+-----+--------------+--------------------+------------------+------------------+------------------+---------+
|                 App|         Category|Rating|Reviews|              Size|   Installs|Type|Price|Content Rating|              Genres|      Last Updated|       Current Ver|       Android Ver|    Ranks|
+--------------------+-----------------+------+-------+------------------+-----------+----+-----+--------------+--------------------+------------------+------------------+------------------+---------+
|U Launcher Lite –...|   ART_AND_DESIGN|     4|  87510|              8.7M| 5,000,000+|Free|    0|      Everyone|        Art & Design|    August 1, 2018|             1.2.4|      4.0.3 and up|High Rank|
|Sketch - Draw & P...|   ART_AND_DESIGN|     4| 215644|               25M|50,000,000+|Free|    0|          Teen|        Art & Design|      June 8, 2018|Varies with device|        4.2 and up|High R

#Task-2: Youtube Videos file, available on Blackboard (under Datasets)

###Q1: Load Youtube Videos file into a dataframe (df2), and

1.   print its schema, and
2.   sample (5) records, and
3.   count number of rows

In [35]:
df2 = spark.read.csv("/USvideos.csv", header=True, inferSchema=True)
df2.printSchema()
df2.show(5)
df2.count()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               t

48137

###Q2: Display only the video title and its tag. However, apply the explode function on tags field so that each tag, sparated by pipe (|), gets listed separately in a separate row.


*   For instance, a video with 2 tags (say: Robots|Boston Dynamics) will end up having two rows, one for "Robots" tag, and another for "Boston Dynamics" tag.



In [36]:
from pyspark.sql.functions import split, explode
df2.withColumn("tags_array", split("tags", "|")) \
   .select("title", explode("tags_array").alias("tag")) \
   .show(truncate=False)

+--------------------------------------------------------------+---+
|title                                                         |tag|
+--------------------------------------------------------------+---+
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |S  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |H  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |A  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |N  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |t  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |e  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |l  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |l  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |   |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |m  |
|WE WANT TO TALK ABOUT OUR MARRIAGE                            |a  |
|WE WANT TO TALK ABOUT OUR MARRIAG

###Q3: Combine the content of two columns "title" and "description", and store it in a new column called "text_info".

In [37]:
from pyspark.sql.functions import concat_ws
df2 = df2.withColumn("text_info", concat_ws(" ", "title", "description"))
df2.select("text_info").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

###Q4: filter the text_info to remove empty lines

In [38]:
df2 = df2.filter(df2["text_info"] != "")
df2.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|           text_info|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|    

###Q5: convert the text_info data to lowercase

In [39]:
from pyspark.sql.functions import lower
df2 = df2.withColumn("text_info", lower("text_info"))
df2.select("text_info").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

###Q6: split the data of text_info field on whitespace (' ') and store it in a new column called "text_info_array"

In [40]:
df2 = df2.withColumn("text_info_array", split("text_info", " "))
df2.select("text_info_array").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

###Q7: Apply explode function on the "text_info_array" field so that each word gets listed on different row.
*   Store the individual words in new column, say "keyword"



In [41]:
df2.select(explode("text_info_array").alias("keyword")).show()

+--------------------+
|             keyword|
+--------------------+
|                  we|
|                want|
|                  to|
|                talk|
|               about|
|                 our|
|            marriage|
|          shantell's|
|             channel|
|                   -|
|https://www.youtu...|
|                   -|
|https://www.loveb...|
|                this|
|               video|
|                  in|
|                  4k|
|                  on|
|                this|
|                  --|
+--------------------+
only showing top 20 rows



###Q8: Group and count occurrence of each word in the "keyword" field from Q7.

In [42]:
df2.select(explode("text_info_array").alias("keyword")) \
   .groupBy("keyword").count().show()

+--------------------+-----+
|             keyword|count|
+--------------------+-----+
|               still| 1721|
|                hope| 1701|
|                 a7s|  179|
|http://www.nbc.co...|  132|
|                some| 5780|
|            debunked|   29|
|            league's|  158|
|              online| 1256|
|             links🍔|  153|
|hellthyjunkfood\n...|   20|
|            adapter:|   40|
|          electrical|   67|
|              freaks|   31|
|               those| 1279|
|               gabi!|   10|
|            affairs,|    1|
|                 art|  755|
|               trail|  138|
|          editors.\n|   57|
|              waters|   40|
+--------------------+-----+
only showing top 20 rows



###Q9: Sort the results by the words in alphanumeric ascending order

In [43]:
df2.select(explode("text_info_array").alias("keyword")) \
   .groupBy("keyword").count() \
   .orderBy("keyword").show()

+--------------------+-----+
|             keyword|count|
+--------------------+-----+
|                    |50608|
|                   !|  246|
|                  !!|    7|
|                 !!!|   27|
|                !!!,|    1|
|   !!!\n\n\npatreon:|   14|
|         !!!\n\nwhat|    6|
|             !!!\nah|    6|
|           !!!\nchad|    6|
|!!!https://goo.gl...|   16|
|         !!!patreon:|    3|
|         !#mammamia2|   21|
|     !\n\n\ndirected|   13|
|!\n\ncoaching:\nk...|    6|
|         !\n\nhere's|    3|
|          !\n\nthank|    3|
|             !\n\nwe|    6|
|!\n______________...|    2|
|              !\nlet|    3|
|           !\nnorman|    5|
+--------------------+-----+
only showing top 20 rows

